# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [5]:
# Initialization

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print(f"OpenAI API Key exists and begins {GOOGLE_API_KEY[:8]}")
else:
    print("OpenAI API Key not set")
    
OLLAMA_BASE_URL = "http://localhost:11434/v1"
LLAMA_MODEL = 'llama3.2'
GEMMA_MODEL = 'gemma3:1b'
PHI_MODEL = 'phi3'
GPT_OSS_MODEL = 'gpt-oss:20b'

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = 'gemini-3.5-flash-lite'

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GOOGLE_API_KEY)

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins AQ.Ab8RN


In [6]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [7]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [8]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [9]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [10]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [11]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [12]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages)
    
    return response.choices[0].message.content

In [14]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Tokyo


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages)
    
    return response.choices[0].message.content

In [19]:
def handle_tool_calls(message):
    responses = []
    print("Handling tool calls: ", message)
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [20]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Handling tool calls:  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_1735523', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function', extra_content={'google': {'thought_signature': 'El4KXAFpFH0TeZ/xAHajDCUjD8kw0r1+C+fI5NQeqANdC4IaDBaqMQ7ZrnhWbs+ObmMylwWlfoAduwSOmPgnqJm66VgBZwz0sDoENlA/JWOe4pIhMAMO8surx3DxstV1'}}), ChatCompletionMessageFunctionToolCall(id='call_1735524', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')])
Tool called for city Tokyo
Tool called for city Berlin


In [21]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [22]:
import sqlite3


In [23]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [24]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [25]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [31]:
def set_ticket_price(destination_city, ticket_price):
    print(f"DATABASE TOOL CALLED: Setting ticket price for {destination_city} to {ticket_price}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (destination_city.lower(), ticket_price, ticket_price))
        result = conn.commit()
        return f"Ticket price for {destination_city} set to {ticket_price}"

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for destination_city, ticket_price in ticket_prices.items():
    set_ticket_price(destination_city, ticket_price)

In [28]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [29]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Handling tool calls:  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_1052291', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function', extra_content={'google': {'thought_signature': 'El4KXAFpFH0TJR84f8CuC6DXUFk/eGkmOM63Q4VWSjqiRCIAzcAd7ookVXuOYJoege51QbfxCI/roUlJzr/91ewgqlaatyMAniGq/5vOyiBG5aE8Z/bqwPq/JFY1ZrYP'}}), ChatCompletionMessageFunctionToolCall(id='call_1052292', function=Function(arguments='{"destination_city":"Yokyo"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_1052293', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')])
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Yokyo
DATABASE TOOL CALLED: Getting price for Berlin
Handling tool calls:  ChatCompletionMessa

## Exercise

Add a tool to set the price of a ticket!

In [ ]:
set_price_tool = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that will have its ticket price registered",
            },
            "ticket_price": {
                "type": "integer",
                "description": "The price of the ticket"
            }
        },
        "required": ["destination_city", "ticket_price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": price_function}, 
    {"type": "function", "function": set_price_tool }
]

In [33]:
def handle_tool_calls(message):
    responses = []
    print("Handling tool calls: ", message)
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')

        if tool_call.function.name == "get_ticket_price":
            response = get_ticket_price(city)
        elif tool_call.function.name == "set_ticket_price":
            price = arguments.get('ticket_price')
            response = set_ticket_price(city, price)
        
        responses.append({
            "role": "tool",
            "content": response,
            "tool_call_id": tool_call.id
        })

    return responses


In [34]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\Dev\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Dev\llm_engineering\.venv\Lib\site-packages\gradio\utils.py", line 882, in async_wrapper
    response = await f(*args, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  Fi

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>